# Chapter 9: Pre-training Paradigms

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch09_pre_training_paradigms.ipynb)


## What is in this notebook, and what it needs before it runs

Five cells, and three of them load a pre-trained checkpoint.

1. **BERT filling a mask**, using left and right context at once, which is the
   thing a causal model cannot do.
2. **Fine-tuning BERT for sentiment** on three examples through the Trainer API.
   Three examples is not a training set; the cell is about the interface.
3. **GPT-2 generating from a prompt**, under causal language modelling.
4. **The GPT-1 to GPT-3 progression as a table.** Pure data, no imports, and the
   one cell here that runs anywhere.
5. **T5 on three tasks through one text-to-text interface.**

Three obstacles on this machine, named here rather than discovered halfway
through. `bert-base-uncased` and `t5-small` are not in the local hub cache and
this build does not download. `T5Tokenizer` needs `sentencepiece`, which is not
installed. And cell 2 runs a fine-tuning loop. So the notebook is committed with
no stored output.

To run it: `pip install sentencepiece`, then run with network access and expect
roughly half a gigabyte of checkpoints the first time. No GPU is needed. Cell 2
takes a couple of minutes on a CPU.

Cell 1 carries its expected answer in a comment: paris at 0.988, then london and
rome far behind. Check it against what the cell actually prints. A number
written in a comment is a claim nobody has rerun, and there are three more of
those in this notebook.


> **This notebook was not executed when it was built, so no cell below has
> stored output.** The reason: cells 1, 2 and 5 need `bert-base-uncased` and `t5-small`, neither of which is in this machine's hub cache, and `T5Tokenizer` additionally needs `sentencepiece`, which is not installed. Cell 2 also runs a fine-tuning loop.
>
> Nothing here is broken. It is code to read now and to run once you have what
> it needs, and the section above says what that is. Build it yourself with
> `python tools/build_notebook.py ch09` on a machine that has them.


### 9.2.2 The Masked Language Modeling Objective

**MLM masking and loss computation with Hugging Face.** The `DataCollatorForLanguageModeling` handles the 80/10/10 strategy automatically; here we show explicit masked prediction for transparency.


In [ ]:
# Colab does not ship these. Running this cell is a no-op if they are already present.
%pip install -q transformers accelerate


In [ ]:
from transformers import BertTokenizer, BertForMaskedLM, pipeline

# BERT predicts masked tokens using full bidirectional context
unmasker = pipeline("fill-mask", model="bert-base-uncased")

# The model sees BOTH left context ("The capital of France is")
# AND right context (".") to predict "paris"
results = unmasker("The capital of France is [MASK].")
for r in results[:3]:
    print(f"  {r['token_str']:>12s}  (score: {r['score']:.3f})")
# Output: paris (0.988), london (0.002), rome (0.001)
# Compare: GPT would see only "The capital of France is" -- no right context


### 9.2.4 Fine-tuning BERT for Downstream Tasks


In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer
from transformers import Trainer, TrainingArguments

# Load pre-trained BERT and tokenizer
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Prepare example training data (sentiment: 0=negative, 1=positive)
train_texts = [
    "This movie was absolutely wonderful and moving.",
    "Terrible film, a complete waste of time.",
    "The acting was superb and the story compelling.",
    "Boring, predictable, and poorly written.",
]
train_labels = [1, 0, 1, 0]

# Tokenize: [CLS] + text + [SEP], padded to 128 tokens
train_encodings = tokenizer(train_texts, truncation=True,
                            padding="max_length", max_length=128,
                            return_tensors="pt")

# Create a simple torch Dataset for the Trainer
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = SentimentDataset(train_encodings, train_labels)

# Fine-tune: 3 epochs at lr=2e-5 -- the standard BERT recipe
args = TrainingArguments(output_dir="./bert-imdb",
    num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=4)
trainer = Trainer(model=model, args=args,
    train_dataset=train_dataset)
trainer.train()  # Illustrative only: real results use ~2,000 IMDB examples; see discussion in text

# Verify the model produces predictions
test_input = tokenizer("A truly excellent performance.",
                       return_tensors="pt")
with torch.no_grad():
    logits = model(**test_input).logits
    pred = torch.argmax(logits, dim=-1).item()
print(f"Prediction: {'positive' if pred == 1 else 'negative'}")
print(f"Logits: {logits.tolist()}")


### 9.3.2 The Causal Language Modeling Objective

**Autoregressive text generation with GPT-2 using top-$p$ (nucleus) sampling.** The model extends the prompt token by token, each step conditioned on all preceding tokens.


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# The prompt IS the task specification -- no fine-tuning required
prompt = "The key insight behind pre-training is that"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Each generated token is conditioned on all previous tokens (CLM)
output = model.generate(input_ids, max_length=80,
    temperature=0.7, do_sample=True,
    top_p=0.9, no_repeat_ngram_size=2)
print(tokenizer.decode(output[0], skip_special_tokens=True))


### 9.3.3 From GPT to GPT-2 to GPT-3: The Power of Scale


In [ ]:
# Illustrating the scale of the GPT family
# No external imports required -- pure data illustration
gpt_models = {
    "GPT-1 (2018)":  {"params": "117M",  "data": "BookCorpus (800M words)",
                      "capability": "Fine-tuning on downstream tasks"},
    "GPT-2 (2019)":  {"params": "1.5B",  "data": "WebText (40GB)",
                      "capability": "Zero-shot task solving"},
    "GPT-3 (2020)":  {"params": "175B",  "data": "300B tokens (mixed)",
                      "capability": "In-context few-shot learning"},
}

# Display the scaling progression
for name, info in gpt_models.items():
    print(f"{name}: {info['params']} params, {info['data']}")
    print(f"  Key capability: {info['capability']}")


### 9.4.1 The Text-to-Text Framework

**T5 text-to-text inference:** the same model, same interface, and same decoding procedure handle translation, summarization, and classification.


In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained("t5-small")
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# Same model, same API, different tasks -- T5's core unification
tasks = [
    "translate English to German: The house is wonderful.",
    "summarize: Pre-training on large corpora enables "
        "transfer learning across NLP tasks.",
    "stsb sentence1: The cat sat. sentence2: The feline rested.",
]
for task in tasks:
    inputs = tokenizer(task, return_tensors="pt",
                       max_length=128, truncation=True)
    outputs = model.generate(**inputs, max_length=50)
    print(f"Input:  {task}")
    print(f"Output: {tokenizer.decode(outputs[0],
                     skip_special_tokens=True)}\n")


---

## Summary

This notebook demonstrated the key code examples from Chapter 9: Pre-training Paradigms. For the full mathematical exposition and discussion, refer to the textbook chapter.
